## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2


# 사전추세(평행추세) 검정 — 긴축기 · 평시

## 왜 이 검정이 필요한가

DiD(이중차분법)에서 `Post×노출도` 계수를 "긴축 때문에 생긴 차이"로 해석하려면, **Post가 없었다면 노출도가 높은 업종과 낮은 업종의 ICR 추세가 원래 평행하게 갔을 것**이라는 가정(평행추세 가정)이 필요하다. 이게 깨지면, 계수에 잡히는 차이가 "긴축 때문"인지 "원래부터 벌어지고 있던 격차의 연장"인지 구분이 안 된다.

그래서 실제 Post가 시작되기 *전* 기간만 따로 떼어, 그 안에서 노출도(또는 현금여력)에 따라 ICR 추세가 이미 갈라지고 있었는지를 아래 방법들로 확인한다.

## 세 가지 방법

- **A. 위약검정(placebo)**: 사전기간 안에서 아무 데나 가짜 Post연도를 그어서, 가짜 Post×노출도가 유의하지 않은지 본다.
- **B. 이벤트 스터디**: 연도더미×노출도를 전부 넣고(기준연도는 실제 Post 직전), 연도별 계수를 각각 확인 + 결합F검정으로 종합 판정한다. **결합F검정이 표준 판정 기준**이다 — 연도를 여러 개 따로 검정하면 다중비교 문제로 우연히 하나쯤 유의하게 나올 수 있기 때문에, 개별 연도 하나 유의한 걸로 "위반"이라 단정하면 과장이다.
- **C. 선형 사전추세**: 사전기간 안에서 (연도)×노출도의 단순 선형 기울기가 0인지만 따로 본다.

## 데이터 구성상 주의

- 긴축기 모형은 실제 Post(2022) 이전 연도를 **2015~2021로 확장**해서 검정한다(패널 자체는 2018~2023뿐이라, 평시 패널의 2015~2017 실측 ICR을 이어붙였다).
- 평시 모형은 사전기간(2015~2017)만으로는 표본이 너무 작아서, ISTANS 수익성 원자료에서 **2010~2014 ICR을 추가로 뽑아 2010~2017로 확장**해서 검정한다.
- 노출도·현금여력은 각 모형 고유의 정의(긴축기=2018~2020 사전평균, 평시=2015~2017 사전평균)를 그대로 쓴다 — 두 트랙을 섞지 않는다.
- 과거 문서의 "2015~17년 p=0.017~0.038 사전추세 위반(팀원B가 발견했다고 서술됨)"은 9/27 전체 프로젝트 감사에서 근거 파일이 전무한 허구로 확인됐다. 이 노트북이 그 자리를 실제로 검증한 것이다.

## ⚠️ 이 노트북 실행 이후에 밝혀진 사실 (미리 안내)

이 노트북의 1차 결과만 보면 "긴축기·평시 모두 결합검정 기준으로는 위반 근거 약함"으로 읽히는데, **이후 두 가지가 이 결론을 바꿔놓았다** (같은 폴더의 `노출도정의_강건성_재현.py`, `2017계수_leaveoneout_재현.py`, `반도체슈퍼사이클_검증.py`에서 실행):

1. **긴축기**: 이 노트북은 사전기간을 2015~2020까지 넓게 잡아서 검정했다(리드 6개). 그런데 **패널 자체가 갖고 있는 연도만(2018·2019·2020, 리드 3개) 좁혀서 검정하면 6개 조합 중 5개에서 실제로 위반이 나온다** — 넓은 창이 옛날의 "깨끗한" 연도를 섞어 넣어서 신호를 희석시켰던 것. 따라서 "긴축기 사전추세는 통과"라는 이 노트북의 1차 결론은 **정정 대상**이다. 2020년 계수(β≈73, p≈0.007~0.011)는 어떤 사양에서도 견고하게 유의하다.
2. **평시**: 이 노트북에서 발견한 "2017년만 유독 이상하다"는 신호는, 나중에 **2017년 반도체 슈퍼사이클(반도체·디스플레이·컴퓨터·통신기기·전지 5개 업종의 ICR 급등)** 때문인 것으로 원인이 완전히 규명됐다(사드 가설은 기각됨). 5개 업종을 빼면 노출도 계수가 β=-117.64→-8.83으로 붕괴한다.

즉 이 노트북은 "사전추세에 뭔가 있다"는 **1차 탐지** 역할을 한 것이고, 정확한 판정과 원인 규명은 뒤에 이어지는 스크립트들이 한다.


## 준비: 두 패널 읽기 + 각 트랙의 노출도·현금여력 추출

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

HERE = PROCESSED
TIGHTENING_CSV = HERE / "재무_원자재_병합패널.csv"
NORMAL_CSV = HERE / "평시_재무_원자재_병합패널.csv"

tight = pd.read_csv(TIGHTENING_CSV, encoding="utf-8-sig")
normal = pd.read_csv(NORMAL_CSV, encoding="utf-8-sig")

# 긴축기 모형의 노출도·현금여력(업종별 상수) 추출
exposure_map_tight = tight.drop_duplicates("업종코드").set_index("업종코드")["사전노출도(%)"].to_dict()
cash_map_tight = tight.drop_duplicates("업종코드").set_index("업종코드")["사전현금여력(%)"].to_dict()

print("긴축기 패널:", len(tight), "행 /", tight["업종코드"].nunique(), "업종")
print("평시 패널:", len(normal), "행 /", normal["업종코드"].nunique(), "업종")


긴축기 패널: 240 행 / 40 업종
평시 패널: 200 행 / 40 업종


---
# Part 1. 긴축기 모형 (Post=2022~2023, 노출도=2018~2020 사전평균)

사전기간(Post=0)을 2015~2021로 확장 — 2015~2017 ICR은 평시 패널에서, 2018~2021은 긴축기 패널(Post=0 행)에서 가져와 이어붙인다.


In [3]:
pre_tight = tight[tight["Post"] == 0][["업종코드", "업종", "연도", "ICR"]].copy()
pre_normal_for_tight = normal[normal["연도"] < 2018][["업종코드", "업종", "연도", "ICR"]].copy()

pre = pd.concat([pre_normal_for_tight, pre_tight], ignore_index=True)
pre["노출도"] = pre["업종코드"].map(exposure_map_tight)
pre["현금여력"] = pre["업종코드"].map(cash_map_tight)
pre = pre.sort_values(["업종코드", "연도"]).reset_index(drop=True)

print("연도 범위:", sorted(pre["연도"].unique()))
print("행 수:", len(pre), "(기대값: 40업종 x 7년 =", 40 * 7, ")")


연도 범위: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
행 수: 280 (기대값: 40업종 x 7년 = 280 )


### A. 위약검정 (가짜 Post=2020~2021, 가짜 사전=2015~2019)

In [4]:
placebo = pre.copy()
placebo["Post_가짜"] = (placebo["연도"] >= 2020).astype(int)
placebo["Post가짜x노출도"] = placebo["Post_가짜"] * placebo["노출도"]

model_a = smf.ols(
    "ICR ~ Post가짜x노출도 + C(업종코드) + C(연도)", data=placebo
).fit(cov_type="cluster", cov_kwds={"groups": placebo["업종코드"]})

coef_a, se_a, p_a = model_a.params["Post가짜x노출도"], model_a.bse["Post가짜x노출도"], model_a.pvalues["Post가짜x노출도"]
print(f"Post가짜×노출도: beta={coef_a:.4f}, SE={se_a:.4f}, p={p_a:.4f}")
print("해석:", "유의(p<0.05) -> 사전추세 위반 의심" if p_a < 0.05 else "비유의 -> 사전추세 위반 근거 없음")


Post가짜×노출도: beta=28.3428, SE=44.6686, p=0.5257
해석: 비유의 -> 사전추세 위반 근거 없음


### B. 이벤트 스터디 (기준연도=2021, 실제 Post 직전) — 노출도

In [5]:
years_sorted = sorted(pre["연도"].unique())
base_year = 2021

formula_terms = " + ".join([f"I((연도=={y})*1) : 노출도" for y in years_sorted if y != base_year])
event_formula = f"ICR ~ {formula_terms} + C(업종코드) + C(연도)"
model_b = smf.ols(event_formula, data=pre).fit(cov_type="cluster", cov_kwds={"groups": pre["업종코드"]})

print(f"{'연도':>6} {'beta':>10} {'SE':>10} {'p-value':>10}")
for y in years_sorted:
    if y == base_year:
        print(f"{y:>6} (기준연도, 0으로 고정)")
        continue
    term = f"I((연도 == {y}) * 1):노출도"
    b, se, p = model_b.params[term], model_b.bse[term], model_b.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {se:>10.4f} {p:>10.4f}")

pre_terms = [f"I((연도 == {y}) * 1):노출도" for y in years_sorted if y != base_year]
hypothesis = " = ".join(pre_terms) + " = 0"
f_test = model_b.f_test(hypothesis)
print()
print(f"결합 F검정(2015~2020 전부 0인가, 6개 리드): F={float(f_test.fvalue):.4f}, p={float(f_test.pvalue):.4f}")
print("해석:", "유의 -> 위반" if float(f_test.pvalue) < 0.05 else "비유의 -> 이 넓은 창 기준으로는 위반 근거 약함(주의: 좁은 창으로 보면 결론이 바뀜, 위 안내 참고)")


    연도       beta         SE    p-value
  2015   -25.6379   124.9512     0.8374
  2016   -22.7922   138.0548     0.8689
  2017    -9.8034    44.9983     0.8275
  2018    30.2848    84.3997     0.7197
  2019   148.7736    64.5219     0.0211
  2020   105.0155    35.9298     0.0035
  2021 (기준연도, 0으로 고정)

결합 F검정(2015~2020 전부 0인가, 6개 리드): F=1.9675, p=0.0941
해석: 비유의 -> 이 넓은 창 기준으로는 위반 근거 약함(주의: 좁은 창으로 보면 결론이 바뀜, 위 안내 참고)


**결과(넓은 창, 6개 리드 2015~2020)**: A p=0.53(비유의), B 결합F p=0.094(비유의). 개별 연도는 2019(p=0.021)·2020(p=0.0035)만 유의. **이 결과는 나중에 좁은 창(2018~2020, 3개 리드)으로 재검증했을 때 뒤집힌다 — 위 안내 참고.**

### C. 선형 사전추세 (노출도)

In [6]:
linear = pre.copy()
linear["연도_중심화"] = linear["연도"] - linear["연도"].mean()
linear["연도x노출도"] = linear["연도_중심화"] * linear["노출도"]

model_c = smf.ols("ICR ~ 연도x노출도 + C(업종코드) + C(연도)", data=linear).fit(
    cov_type="cluster", cov_kwds={"groups": linear["업종코드"]}
)
coef_c, se_c, p_c = model_c.params["연도x노출도"], model_c.bse["연도x노출도"], model_c.pvalues["연도x노출도"]
print(f"연도x노출도(선형기울기): beta={coef_c:.4f}, SE={se_c:.4f}, p={p_c:.4f}")
print("해석:", "유의 -> 선형추세 존재" if p_c < 0.05 else "비유의 -> 선형추세 근거 없음")


연도x노출도(선형기울기): beta=17.5395, SE=25.1500, p=0.4856
해석: 비유의 -> 선형추세 근거 없음


### D~F. 위 A~C를 노출도 대신 현금여력 기준으로 반복

DiD 핵심변수가 노출도×현금여력 둘 다이므로, 현금여력 쪽도 평행추세를 확인해야 한다.

In [7]:
# D. 위약검정 (현금여력)
placebo["Post가짜x현금여력"] = placebo["Post_가짜"] * placebo["현금여력"]
model_d = smf.ols(
    "ICR ~ Post가짜x현금여력 + C(업종코드) + C(연도)", data=placebo
).fit(cov_type="cluster", cov_kwds={"groups": placebo["업종코드"]})
coef_d, se_d, p_d = model_d.params["Post가짜x현금여력"], model_d.bse["Post가짜x현금여력"], model_d.pvalues["Post가짜x현금여력"]
print(f"D. Post가짜×현금여력: beta={coef_d:.4f}, SE={se_d:.4f}, p={p_d:.4f}")


D. Post가짜×현금여력: beta=-6.5591, SE=9.7519, p=0.5012


In [8]:
# E. 이벤트 스터디 (현금여력)
formula_terms_cb = " + ".join([f"I((연도=={y})*1) : 현금여력" for y in years_sorted if y != base_year])
event_formula_cb = f"ICR ~ {formula_terms_cb} + C(업종코드) + C(연도)"
model_e = smf.ols(event_formula_cb, data=pre).fit(cov_type="cluster", cov_kwds={"groups": pre["업종코드"]})

print(f"{'연도':>6} {'beta':>10} {'SE':>10} {'p-value':>10}")
for y in years_sorted:
    if y == base_year:
        print(f"{y:>6} (기준연도)")
        continue
    term = f"I((연도 == {y}) * 1):현금여력"
    b, se, p = model_e.params[term], model_e.bse[term], model_e.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {se:>10.4f} {p:>10.4f}")

pre_terms_cb = [f"I((연도 == {y}) * 1):현금여력" for y in years_sorted if y != base_year]
hypothesis_cb = " = ".join(pre_terms_cb) + " = 0"
f_test_cb = model_e.f_test(hypothesis_cb)
print()
print(f"E. 결합 F검정: F={float(f_test_cb.fvalue):.4f}, p={float(f_test_cb.pvalue):.4f}")


    연도       beta         SE    p-value
  2015    20.3224    29.4190     0.4897
  2016    22.2223    32.2777     0.4912
  2017    -2.0854     6.5184     0.7490
  2018   -16.2859    14.3859     0.2576
  2019   -23.9696    12.6554     0.0582
  2020   -13.0367     7.4310     0.0794
  2021 (기준연도)

E. 결합 F검정: F=1.5891, p=0.1763


In [9]:
# F. 선형 사전추세 (현금여력)
linear["연도x현금여력"] = linear["연도_중심화"] * linear["현금여력"]
model_f = smf.ols(
    "ICR ~ 연도x현금여력 + C(업종코드) + C(연도)", data=linear
).fit(cov_type="cluster", cov_kwds={"groups": linear["업종코드"]})
coef_f, se_f, p_f = model_f.params["연도x현금여력"], model_f.bse["연도x현금여력"], model_f.pvalues["연도x현금여력"]
print(f"F. 연도x현금여력(선형기울기): beta={coef_f:.4f}, SE={se_f:.4f}, p={p_f:.4f}")


F. 연도x현금여력(선형기울기): beta=-5.4775, SE=6.1188, p=0.3707


**결과(현금여력)**: D p=0.50, E 결합F p=0.176, F p=0.37 — 전부 비유의. 노출도보다도 신호가 약함.

---
# Part 2. 평시 모형 (Post=2018~2019, 노출도=2015~2017 사전평균)

사전기간을 2010~2017로 확장 — ISTANS 수익성 원자료(2010~2025 커버)에서 2010~2014 ICR을 추가로 뽑는다.


In [10]:
import re
import openpyxl

DATA = RAW
PROFIT_XLSX = DATA / "ISTANS 수익성" / "수익성_20260919005811.xlsx"


def clean_var_name(raw):
    m = re.match(r"\d+\s+(.+?)\s+\(", str(raw))
    return m.group(1).strip() if m else str(raw).strip()


def read_istans_sheet(path, sheet="데이터"):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet]
    rows = list(ws.iter_rows(min_row=1, max_row=200, max_col=400, values_only=True))
    year_row, var_row = rows[0], rows[1]
    years, variables = [], []
    for c in range(1, len(year_row)):
        ym = re.search(r"(\d{4})", str(year_row[c])) if year_row[c] else None
        years.append(int(ym.group(1)) if ym else None)
        variables.append(clean_var_name(var_row[c]) if var_row[c] else None)
    data = {}
    for r in range(2, len(rows)):
        raw_name = rows[r][0]
        if raw_name is None:
            continue
        m = re.search(r"(I\d+)\s+(.+)", str(raw_name).strip())
        if not m:
            continue
        code_ = m.group(1)
        if not re.fullmatch(r"I3[1-4]\d{2}", code_):
            continue
        for c in range(1, len(rows[r])):
            if years[c - 1] is None or variables[c - 1] is None:
                continue
            data[(code_, years[c - 1], variables[c - 1])] = rows[r][c]
    return data


print("수익성 원자료(2010~2025)에서 2010~2014 ICR 추출 중...")
profit_data = read_istans_sheet(PROFIT_XLSX)

exposure_map = normal.drop_duplicates("업종코드").set_index("업종코드")["사전노출도(%)"].to_dict()
cash_map = normal.drop_duplicates("업종코드").set_index("업종코드")["사전현금여력(%)"].to_dict()
industry_names = normal.drop_duplicates("업종코드").set_index("업종코드")["업종"].to_dict()
industry_codes = sorted(exposure_map.keys())

extra_rows = []
missing = []
for code_ in industry_codes:
    for y in range(2010, 2015):
        icr = profit_data.get((code_, y, "이자보상비율"))
        if icr is None:
            missing.append((code_, y))
        extra_rows.append({"업종코드": code_, "업종": industry_names[code_], "연도": y, "ICR": icr})
print("2010~2014 ICR 결측:", missing or "없음")

extra = pd.DataFrame(extra_rows)
pre_2015_2017 = normal[normal["연도"] < 2018][["업종코드", "업종", "연도", "ICR"]].copy()
pre_n = pd.concat([extra, pre_2015_2017], ignore_index=True)
pre_n["노출도"] = pre_n["업종코드"].map(exposure_map)
pre_n["현금여력"] = pre_n["업종코드"].map(cash_map)
pre_n = pre_n.sort_values(["업종코드", "연도"]).reset_index(drop=True)

# 담배(I3402)는 2010~2017 내내 ICR이 만~3만대(다른 업종 300~800대)로 극단치
EXCLUDE_OUTLIER = "I3402"
pre_n_excl = pre_n[pre_n["업종코드"] != EXCLUDE_OUTLIER].copy()

print("연도 범위:", sorted(pre_n["연도"].unique()))
print("행 수:", len(pre_n), "(기대값: 40업종 x 8년 =", 40 * 8, ")")


수익성 원자료(2010~2025)에서 2010~2014 ICR 추출 중...
2010~2014 ICR 결측: 없음
연도 범위: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
행 수: 320 (기대값: 40업종 x 8년 = 320 )


<python-env> UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### A. 위약검정 (가짜 Post=2016~2017, 가짜 사전=2010~2015)

In [11]:
placebo_n = pre_n.copy()
placebo_n["Post_가짜"] = (placebo_n["연도"] >= 2016).astype(int)
placebo_n["Post가짜x노출도"] = placebo_n["Post_가짜"] * placebo_n["노출도"]

model_a_n = smf.ols(
    "ICR ~ Post가짜x노출도 + C(업종코드) + C(연도)", data=placebo_n
).fit(cov_type="cluster", cov_kwds={"groups": placebo_n["업종코드"]})
coef_a_n, se_a_n, p_a_n = model_a_n.params["Post가짜x노출도"], model_a_n.bse["Post가짜x노출도"], model_a_n.pvalues["Post가짜x노출도"]
print(f"Post가짜×노출도: beta={coef_a_n:.4f}, SE={se_a_n:.4f}, p={p_a_n:.4f}")


Post가짜×노출도: beta=-65.5898, SE=56.3144, p=0.2441

### B / B'. 이벤트 스터디 (기준연도=2017) — 노출도, 전체표본 vs 담배(I3402) 제외

In [12]:
years_sorted_n = sorted(pre_n["연도"].unique())
base_year_n = 2017
formula_terms_n = " + ".join([f"I((연도=={y})*1) : 노출도" for y in years_sorted_n if y != base_year_n])
event_formula_n = f"ICR ~ {formula_terms_n} + C(업종코드) + C(연도)"

model_b_n = smf.ols(event_formula_n, data=pre_n).fit(cov_type="cluster", cov_kwds={"groups": pre_n["업종코드"]})
print("--- B. 전체표본(40업종) ---")
for y in years_sorted_n:
    if y == base_year_n:
        continue
    term = f"I((연도 == {y}) * 1):노출도"
    b, se, p = model_b_n.params[term], model_b_n.bse[term], model_b_n.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {p:>10.4f}")
pre_terms_n = [f"I((연도 == {y}) * 1):노출도" for y in years_sorted_n if y != base_year_n]
hyp_n = " = ".join(pre_terms_n) + " = 0"
f_b = model_b_n.f_test(hyp_n)
print(f"결합F(40업종): F={float(f_b.fvalue):.4f}, p={float(f_b.pvalue):.4f}")

model_b2_n = smf.ols(event_formula_n, data=pre_n_excl).fit(cov_type="cluster", cov_kwds={"groups": pre_n_excl["업종코드"]})
print("\n--- B'. 담배(I3402) 제외 ---")
for y in years_sorted_n:
    if y == base_year_n:
        continue
    term = f"I((연도 == {y}) * 1):노출도"
    b, se, p = model_b2_n.params[term], model_b2_n.bse[term], model_b2_n.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {p:>10.4f}")
f_b2 = model_b2_n.f_test(hyp_n)
print(f"결합F(담배제외): F={float(f_b2.fvalue):.4f}, p={float(f_b2.pvalue):.4f}")


--- B. 전체표본(40업종) ---
  2010     9.5493     0.9123
  2011    84.7197     0.2347
  2012    39.6775     0.5670
  2013    46.6712     0.4052
  2014    48.5424     0.5373
  2015   -85.9385     0.6571
  2016   -83.4390     0.6946
결합F(40업종): F=2.5609, p=0.0285

--- B'. 담배(I3402) 제외 ---
  2010    36.7458     0.6877
  2011    89.7817     0.2592
  2012    77.9230     0.2092
  2013    76.3227     0.1348
  2014   114.3549     0.0056
  2015    98.2448     0.0070
  2016   117.6381     0.0137
결합F(담배제외): F=1.9430, p=0.0894


**결과**: 결합F검정 40업종 p=0.0285(유의) → 담배 제외 시 p=0.089(비유의)로 전환. 담배(I3402)가 2010~2017 내내 ICR 만~3만대인 극단치라서 최초 유의성이 왜곡됐던 것.

### C. 선형 사전추세 (노출도)

In [13]:
linear_n = pre_n.copy()
linear_n["연도_중심화"] = linear_n["연도"] - linear_n["연도"].mean()
linear_n["연도x노출도"] = linear_n["연도_중심화"] * linear_n["노출도"]
model_c_n = smf.ols("ICR ~ 연도x노출도 + C(업종코드) + C(연도)", data=linear_n).fit(
    cov_type="cluster", cov_kwds={"groups": linear_n["업종코드"]}
)
coef_c_n, se_c_n, p_c_n = model_c_n.params["연도x노출도"], model_c_n.bse["연도x노출도"], model_c_n.pvalues["연도x노출도"]
print(f"연도x노출도(선형기울기): beta={coef_c_n:.4f}, SE={se_c_n:.4f}, p={p_c_n:.4f}")


연도x노출도(선형기울기): beta=-15.2692, SE=18.4540, p=0.4080


### D~F. 현금여력 기준 (D=A, E/E'=B/B', F=C)

In [14]:
# D. 위약검정 (현금여력)
placebo_n["Post가짜x현금여력"] = placebo_n["Post_가짜"] * placebo_n["현금여력"]
model_d_n = smf.ols(
    "ICR ~ Post가짜x현금여력 + C(업종코드) + C(연도)", data=placebo_n
).fit(cov_type="cluster", cov_kwds={"groups": placebo_n["업종코드"]})
coef_d_n, se_d_n, p_d_n = model_d_n.params["Post가짜x현금여력"], model_d_n.bse["Post가짜x현금여력"], model_d_n.pvalues["Post가짜x현금여력"]
print(f"D. Post가짜×현금여력: beta={coef_d_n:.4f}, p={p_d_n:.4f}")


D. Post가짜×현금여력: beta=15.1163, p=0.0635


In [15]:
# E/E'. 이벤트 스터디 (현금여력, 기준연도=2017)
formula_terms_cb_n = " + ".join([f"I((연도=={y})*1) : 현금여력" for y in years_sorted_n if y != base_year_n])
event_formula_cb_n = f"ICR ~ {formula_terms_cb_n} + C(업종코드) + C(연도)"

model_e_n = smf.ols(event_formula_cb_n, data=pre_n).fit(cov_type="cluster", cov_kwds={"groups": pre_n["업종코드"]})
pre_terms_cb_n = [f"I((연도 == {y}) * 1):현금여력" for y in years_sorted_n if y != base_year_n]
hyp_cb_n = " = ".join(pre_terms_cb_n) + " = 0"
f_e_n = model_e_n.f_test(hyp_cb_n)
print(f"E. 결합F(전체표본): F={float(f_e_n.fvalue):.4f}, p={float(f_e_n.pvalue):.4f}")

model_e2_n = smf.ols(event_formula_cb_n, data=pre_n_excl).fit(cov_type="cluster", cov_kwds={"groups": pre_n_excl["업종코드"]})
print("\n--- E'. 담배 제외, 연도별 계수 ---")
for y in years_sorted_n:
    if y == base_year_n:
        continue
    term = f"I((연도 == {y}) * 1):현금여력"
    b, se, p = model_e2_n.params[term], model_e2_n.bse[term], model_e2_n.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {p:>10.4f}")
f_e2_n = model_e2_n.f_test(hyp_cb_n)
print(f"E'. 결합F(담배제외): F={float(f_e2_n.fvalue):.4f}, p={float(f_e2_n.pvalue):.4f}")


E. 결합F(전체표본): F=1.4432, p=0.2163



--- E'. 담배 제외, 연도별 계수 ---
  2010   -14.9715     0.0240
  2011   -18.9303     0.0258
  2012   -13.9858     0.0284
  2013   -11.6784     0.0460
  2014   -14.5812     0.0159
  2015   -12.5006     0.0185
  2016   -13.8425     0.0247
E'. 결합F(담배제외): F=0.9541, p=0.4778


**결과**: 담배 포함 결합F p=0.216(비유의)인데, 담배 제외(E')하면 **개별 연도(2010~2016)가 전부 p=0.016~0.046로 유의**한데 결합F는 p=0.478(비유의)이라는 모순적 패턴이 나타남 — 계수가 다 비슷한 크기·방향이라 "기준연도로 쓴 2017년 자체가 이상치인 것 아니냐"는 가설을 세우게 됨(아래 G 섹션에서 검증).

In [16]:
# F. 선형 사전추세 (현금여력)
linear_n["연도x현금여력"] = linear_n["연도_중심화"] * linear_n["현금여력"]
model_f_n = smf.ols(
    "ICR ~ 연도x현금여력 + C(업종코드) + C(연도)", data=linear_n
).fit(cov_type="cluster", cov_kwds={"groups": linear_n["업종코드"]})
coef_f_n, se_f_n, p_f_n = model_f_n.params["연도x현금여력"], model_f_n.bse["연도x현금여력"], model_f_n.pvalues["연도x현금여력"]
print(f"F. 연도x현금여력(선형기울기): beta={coef_f_n:.4f}, p={p_f_n:.4f}")


F. 연도x현금여력(선형기울기): beta=4.2761, p=0.1211


### G / G'. 가설 검증 — 기준연도를 2017 → 2016으로 교체

2017년이 진짜 이상치라면: 기준을 2016으로 바꿨을 때 "2017 vs 2016" 비교 항만 유의하고 나머지(2010~2015)는 비유의해야 한다. 반대로 진짜 다년간 추세라면, 기준을 바꿔도 여러 연도가 유의한 패턴이 유지돼야 한다.

In [17]:
base_year_2016 = 2016
formula_terms_2016 = " + ".join([f"I((연도=={y})*1) : 현금여력" for y in years_sorted_n if y != base_year_2016])
event_formula_2016 = f"ICR ~ {formula_terms_2016} + C(업종코드) + C(연도)"

model_g2 = smf.ols(event_formula_2016, data=pre_n_excl).fit(cov_type="cluster", cov_kwds={"groups": pre_n_excl["업종코드"]})
print("G'. 담배 제외, 기준연도=2016")
for y in years_sorted_n:
    if y == base_year_2016:
        print(f"{y:>6} (기준연도)")
        continue
    term = f"I((연도 == {y}) * 1):현금여력"
    b, se, p = model_g2.params[term], model_g2.bse[term], model_g2.pvalues[term]
    print(f"{y:>6} {b:>10.4f} {p:>10.4f}")


G'. 담배 제외, 기준연도=2016
  2010    -1.1290     0.7579
  2011    -5.0879     0.2789
  2012    -0.1433     0.9696
  2013     2.1640     0.5875
  2014    -0.7387     0.7816
  2015     1.3419     0.4708
  2016 (기준연도)
  2017    13.8425     0.0247


**결과**: 2010~2015는 전부 비유의(p=0.28~0.97), **2017만 유일하게 유의(p=0.025)**.

**최종 결론**: "2017년 자체가 이상치"라는 가설이 확인됨 — 다년간 추세가 아니라 2017년 단일 이벤트성 이상이다. (이후 별도 스크립트에서 이 2017년 이상이 반도체 슈퍼사이클 때문임이 완전히 규명됨 — 노트북 도입부의 안내 참고.)


---
# 종합 결론 (이 노트북 실행 시점 기준, 이후 정정사항은 도입부 참고)

| | 긴축기(노출도) | 긴축기(현금여력) | 평시(노출도) | 평시(현금여력) |
|---|---|---|---|---|
| 위약검정 | p=0.53 | p=0.50 | p=0.24 | p=0.06 |
| 이벤트스터디 결합F | p=0.094 | p=0.176 | p=0.089(담배제외) | p=0.478(담배제외) |
| 선형추세 | p=0.49 | p=0.37 | p=0.41 | p=0.12 |

**정직한 결론**: "평행추세 완전 통과"라 단정할 수 없다. 이 노트북의 넓은-창 결과만 보면 위반 근거가 약해 보이지만, 좁은 창(패널 자체 리드만)으로 보면 긴축기도 실제로 위반 신호가 있고, 평시 2017년은 반도체 슈퍼사이클로 원인까지 규명됐다. **최종적으로는 "Post 직전 1~2년 구간(긴축기 2019~2020)과 2017년 반도체 이상치는 논문 한계 섹션에 명시"가 맞는 서술이다.**


---
# 부록 1. 노출도 정의 강건성 재현 (팀원 결과 검증)

**한 내용**: 팀원이 공유한 "노출도 정의를 2018~20/2015~17/2015~16 세 가지로 바꿔도 본분석·사전추세 결과가 비슷한가" 표를 실제로 재현해서 검증한다.

**왜 필요했나**: 팀원이 보낸 사전추세 F검정 표에서 "2018~20, 40업종: F=3.223, p=0.033, 위반"이라는 결과가, 본문(Part 1)에서 내가 넓은 창(2015~2020 리드 6개)으로 돌린 결과(F=1.9675, p=0.094, 비유의)와 정면으로 달랐다. 그래서 그대로 믿지 않고 직접 재현했다.

**과정**: 노출도를 (1)2018~20 원래 정의 (2)2015~17 (3)2015~16(2017 제외) 세 가지로 계산하고, 각각에 대해 A.요약통계 B.본분석(Post×노출도, 긴축기 패널 2018~2023 전체) C.사전추세 F검정(**단, 이번엔 패널 자체 연도만 사용 — 리드를 2018·2019·2020 세 개로 좁힘**, 팀원이 쓴 것으로 추정되는 방식)을 40업종/담배제외 두 표본으로 반복한다.


In [18]:
import numpy as np

STAB_2010_2019_APP = DATA / "ISTANS 안정성" / "안정성_20260919005109.xlsx"


def read_istans_sheet_full(path, sheet="데이터"):
    # Part 2에서 정의한 read_istans_sheet는 이름 없이 data만 반환하는 버전이라,
    # 여기서는 이름까지 함께 반환하는 버전을 별도로 새로 정의한다.
    import openpyxl as _openpyxl
    wb = _openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet]
    rows = list(ws.iter_rows(min_row=1, max_row=200, max_col=400, values_only=True))
    year_row, var_row = rows[0], rows[1]
    years, variables = [], []
    for c in range(1, len(year_row)):
        ym = re.search(r"(\d{4})", str(year_row[c])) if year_row[c] else None
        years.append(int(ym.group(1)) if ym else None)
        variables.append(clean_var_name(var_row[c]) if var_row[c] else None)
    data = {}
    names = {}
    for r in range(2, len(rows)):
        raw_name = rows[r][0]
        if raw_name is None:
            continue
        m = re.search(r"(I\d+)\s+(.+)", str(raw_name).strip())
        if not m:
            continue
        code_, name_ = m.group(1), m.group(2).strip()
        if not re.fullmatch(r"I3[1-4]\d{2}", code_):
            continue
        names[code_] = name_
        for c in range(1, len(rows[r])):
            if years[c-1] is None or variables[c-1] is None:
                continue
            data[(code_, years[c-1], variables[c-1])] = rows[r][c]
    return data, names


print("안정성 원자료(2010~2019) 읽는 중...")
stab_data_app, stab_names_app = read_istans_sheet_full(STAB_2010_2019_APP)
industry_codes_app = sorted(stab_names_app.keys())


def compute_exposure(years):
    result = {}
    for code_ in industry_codes_app:
        vals = []
        for y in years:
            assets = stab_data_app.get((code_, y, "자산총계"))
            sb = stab_data_app.get((code_, y, "단기차입"))
            sd = stab_data_app.get((code_, y, "단기사채"))
            if assets is None or sb is None or sd is None or assets == 0:
                continue
            vals.append((sb + sd) / assets)
        result[code_] = (sum(vals) / len(vals) * 100) if len(vals) == len(years) else None
    return result


exposure_2018_2020 = tight.drop_duplicates("업종코드").set_index("업종코드")["사전노출도(%)"].to_dict()
exposure_2015_2017 = compute_exposure([2015, 2016, 2017])
exposure_2015_2016 = compute_exposure([2015, 2016])

definitions = {
    "2018~20 (원래)": exposure_2018_2020,
    "2015~17": exposure_2015_2017,
    "2015~16 (2017제외)": exposure_2015_2016,
}

print()
print("A. 노출도 정의 3종 요약통계")
print(f"{'정의':>20} {'평균':>8} {'SD':>8} {'최소':>8} {'최대':>8}")
for name, m in definitions.items():
    vals = np.array(list(m.values()), dtype=float)
    print(f"{name:>20} {vals.mean():>8.2f} {vals.std(ddof=1):>8.2f} {vals.min():>8.2f} {vals.max():>8.2f}")


안정성 원자료(2010~2019) 읽는 중...


A. 노출도 정의 3종 요약통계
                  정의       평균       SD       최소       최대
        2018~20 (원래)     9.88     4.23     2.12    21.44
             2015~17    10.45     4.74     2.17    21.97
    2015~16 (2017제외)    10.53     4.85     1.54    22.12


<python-env> UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [19]:
EXCLUDE_OUTLIER_APP = "I3402"  # 담배

print()
print("B. 본분석 Post×노출도 (긴축기 패널 2018~2023 전체)")
print(f"{'정의':>20} {'표본':>10} {'beta':>10} {'SE':>10} {'p값':>10}")
for name, m in definitions.items():
    for sample_name, df_sample in [("40업종", tight), ("담배제외", tight[tight["업종코드"] != EXCLUDE_OUTLIER_APP])]:
        df = df_sample.copy()
        df["노출도"] = df["업종코드"].map(m)
        df["Postx노출도"] = df["Post"] * df["노출도"]
        model = smf.ols("ICR ~ Postx노출도 + C(업종코드) + C(연도)", data=df).fit(
            cov_type="cluster", cov_kwds={"groups": df["업종코드"]}
        )
        b, se, p = model.params["Postx노출도"], model.bse["Postx노출도"], model.pvalues["Postx노출도"]
        print(f"{name:>20} {sample_name:>10} {b:>10.2f} {se:>10.2f} {p:>10.3f}")



B. 본분석 Post×노출도 (긴축기 패널 2018~2023 전체)
                  정의         표본       beta         SE         p값
        2018~20 (원래)       40업종      53.66      40.20      0.182
        2018~20 (원래)       담배제외      65.81      38.88      0.091
             2015~17       40업종      37.51      38.53      0.330
             2015~17       담배제외      57.48      35.73      0.108


    2015~16 (2017제외)       40업종      37.02      38.73      0.339
    2015~16 (2017제외)       담배제외      58.22      35.71      0.103


In [20]:
print()
print("C. 사전추세 F검정 (긴축기 패널 자체 연도만, 리드=2018·2019·2020, 기준연도=2021)")
print(f"{'정의':>20} {'표본':>10} {'F':>10} {'p값':>10} {'판정':>8}")
lead_years_app = [2018, 2019, 2020]
base_year_app = 2021
all_years_app = sorted(tight["연도"].unique())

for name, m in definitions.items():
    for sample_name, df_sample in [("40업종", tight), ("담배제외", tight[tight["업종코드"] != EXCLUDE_OUTLIER_APP])]:
        df = df_sample.copy()
        df["노출도"] = df["업종코드"].map(m)
        formula_terms = " + ".join([f"I((연도=={y})*1) : 노출도" for y in all_years_app if y != base_year_app])
        formula = f"ICR ~ {formula_terms} + C(업종코드) + C(연도)"
        model = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["업종코드"]})
        lead_terms = [f"I((연도 == {y}) * 1):노출도" for y in lead_years_app]
        hyp = " = ".join(lead_terms) + " = 0"
        f_test = model.f_test(hyp)
        F, p = float(f_test.fvalue), float(f_test.pvalue)
        verdict = "위반" if p < 0.05 else ("경계" if p < 0.10 else "통과")
        print(f"{name:>20} {sample_name:>10} {F:>10.3f} {p:>10.3f} {verdict:>8}")

print()
print("D. 가장 보수적 조합(2015~16 노출도 + 담배제외) 연도별 계수")
df = tight[tight["업종코드"] != EXCLUDE_OUTLIER_APP].copy()
df["노출도"] = df["업종코드"].map(exposure_2015_2016)
formula_terms = " + ".join([f"I((연도=={y})*1) : 노출도" for y in all_years_app if y != base_year_app])
formula = f"ICR ~ {formula_terms} + C(업종코드) + C(연도)"
model = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["업종코드"]})
print(f"{'연도':>6} {'beta':>10} {'p값':>10}")
for y in all_years_app:
    if y == base_year_app:
        print(f"{y:>6} (기준연도)")
        continue
    term = f"I((연도 == {y}) * 1):노출도"
    b, p = model.params[term], model.pvalues[term]
    print(f"{y:>6} {b:>10.2f} {p:>10.3f}")



C. 사전추세 F검정 (긴축기 패널 자체 연도만, 리드=2018·2019·2020, 기준연도=2021)
                  정의         표본          F         p값       판정
        2018~20 (원래)       40업종      3.223      0.033       위반
        2018~20 (원래)       담배제외      2.927      0.046       위반


             2015~17       40업종      3.413      0.027       위반


             2015~17       담배제외      2.888      0.048       위반
    2015~16 (2017제외)       40업종      3.274      0.031       위반
    2015~16 (2017제외)       담배제외      2.557      0.069       경계

D. 가장 보수적 조합(2015~16 노출도 + 담배제외) 연도별 계수
    연도       beta         p값
  2018     -20.96      0.697
  2019      92.73      0.052
  2020      73.29      0.007
  2021 (기준연도)
  2022      62.79      0.042
  2023     126.18      0.061


**결과(재현값, 팀원 원값과 거의 정확히 일치)**:

| 정의 | 표본 | F | p | 판정 |
|---|---|---|---|---|
| 2018~20 | 40업종 | 3.223 | 0.033 | 위반 |
| 2018~20 | 담배제외 | 2.927 | 0.046 | 위반 |
| 2015~17 | 40업종 | 3.413 | 0.027 | 위반 |
| 2015~17 | 담배제외 | 2.888 | 0.048 | 위반 |
| 2015~16 | 40업종 | 3.274 | 0.031 | 위반 |
| 2015~16 | 담배제외 | 2.557 | 0.069 | 경계 |

**해석**: 팀원 수치가 실제 재현됨 — 허구가 아니라 진짜 계산이었다. 본문(Part 1)의 "긴축기 사전추세는 결합F 비유의(통과)"라는 결론은 **넓은 창(2015~2020, 리드 6개)을 썼기 때문**이고, 패널 자체 연도만(리드 3개) 쓰는 더 표준적인 방식으로 보면 6개 조합 중 5개에서 위반이 나온다. **이 노트북 Part 1의 결론은 이 부록으로 정정된다.** 다만 노출도 정의 자체(2018~20 vs 2015~17 vs 2015~16)는 본분석 결과(β 37~66)에 거의 영향을 주지 않는다 — 영향은 담배 제외 여부가 훨씬 크다. 2020년 계수(β≈73, p≈0.007~0.011)는 모든 조합에서 견고하게 유의하다.


---
# 부록 2. 2017 계수 leave-one-out 재현 (팀원 결과 검증)

**한 내용**: 팀원이 공유한 "평시 노출도 이벤트스터디에서 담배 제외 후 기준연도=2016으로 두면 2017 계수=-116.66(p=0.028)이고, 39업종에서 하나씩 더 빼도(leave-one-out) 부호가 항상 음수, 35/39건에서 5% 유의 유지"라는 주장을 재현 검증한다.

**과정**: 담배 제외 39업종에서 2017 계수를 먼저 확인하고, 그 39업종에서 다시 하나씩 빼가며(38개 업종 표본) 2017 계수를 재추정한다.


In [21]:
normal_app = pd.read_csv(HERE / "평시_재무_원자재_병합패널.csv", encoding="utf-8-sig")
exposure_map_app = normal_app.drop_duplicates("업종코드").set_index("업종코드")["사전노출도(%)"].to_dict()
industry_names_app = normal_app.drop_duplicates("업종코드").set_index("업종코드")["업종"].to_dict()
industry_codes_app2 = sorted(exposure_map_app.keys())

extra_rows_app = []
for code_ in industry_codes_app2:
    for y in range(2010, 2015):
        icr = profit_data.get((code_, y, "이자보상비율"))
        extra_rows_app.append({"업종코드": code_, "업종": industry_names_app[code_], "연도": y, "ICR": icr})
extra_app = pd.DataFrame(extra_rows_app)
pre_2015_2017_app = normal_app[normal_app["연도"] < 2018][["업종코드", "업종", "연도", "ICR"]].copy()
pre_app = pd.concat([extra_app, pre_2015_2017_app], ignore_index=True)
pre_app["노출도"] = pre_app["업종코드"].map(exposure_map_app)
pre_app = pre_app.sort_values(["업종코드", "연도"]).reset_index(drop=True)

years_sorted_app = sorted(pre_app["연도"].unique())
base_year_2016_app = 2016
pre_excl_tobacco_app = pre_app[pre_app["업종코드"] != EXCLUDE_OUTLIER_APP].copy()


def get_2017_coef(df):
    formula_terms = " + ".join([f"I((연도=={y})*1) : 노출도" for y in years_sorted_app if y != base_year_2016_app])
    formula = f"ICR ~ {formula_terms} + C(업종코드) + C(연도)"
    model = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["업종코드"]})
    term = "I((연도 == 2017) * 1):노출도"
    return model.params[term], model.pvalues[term]


b0, p0 = get_2017_coef(pre_excl_tobacco_app)
print(f"담배 제외(39업종), 기준연도=2016, 2017 계수: beta={b0:.2f}, p={p0:.3f}")


담배 제외(39업종), 기준연도=2016, 2017 계수: beta=-117.64, p=0.014


In [22]:
results = []
for code_ in industry_codes_app2:
    if code_ == EXCLUDE_OUTLIER_APP:
        continue
    df = pre_excl_tobacco_app[pre_excl_tobacco_app["업종코드"] != code_]
    b, p = get_2017_coef(df)
    results.append({"제외업종": code_, "업종명": industry_names_app[code_], "beta": b, "p": p})

res_df = pd.DataFrame(results)
print(f"leave-one-out 실행 건수: {len(res_df)}")
print(f"beta 범위: {res_df['beta'].min():.1f} ~ {res_df['beta'].max():.1f}")
print(f"부호(음수) 유지 건수: {(res_df['beta'] < 0).sum()}/{len(res_df)}")
print(f"5% 유의 유지 건수: {(res_df['p'] < 0.05).sum()}/{len(res_df)}")
res_df["abs_change"] = (res_df["beta"] - b0).abs()
print()
print("영향 큰 순 상위 5개:")
print(res_df.sort_values("abs_change", ascending=False).head(5)[["제외업종", "업종명", "beta", "p"]].to_string(index=False))


leave-one-out 실행 건수: 39
beta 범위: -135.2 ~ -91.2
부호(음수) 유지 건수: 39/39
5% 유의 유지 건수: 39/39

영향 큰 순 상위 5개:
 제외업종   업종명        beta        p
I3102   반도체  -91.179157 0.041414
I3108    전지  -96.435069 0.037831
I3310    주조 -135.212630 0.008246
I3105  통신기기 -101.820512 0.035846
I3103 디스플레이 -103.139769 0.041734


**결과(재현값, 팀원 원값과 거의 일치)**: 전체(39업종) β=-117.64, p=0.014 (팀원 -116.66, p=0.028). leave-one-out 범위 -135.2~-91.2 (팀원 -133.6~-91.3), 39/39 부호유지. 영향이 가장 큰 업종: 반도체(빼면 -91.2로 가장 약해짐), 주조(빼면 -135.2로 가장 강해짐).

**해석**: 특정 업종 하나의 문제가 아니라 견고한 패턴 — 그렇다고 원인이 밝혀진 건 아니다(다음 부록에서 원인 규명). 이 시점에는 "2017년에 노출도 높은 업종의 ICR이 유의하게 낮았고, 이건 leave-one-out으로도 견고하다"까지만 확인된 것이고, 왜 그런지는 아직 모른다.


---
# 부록 3. 반도체 슈퍼사이클 검증 — 2017 계수의 실제 원인 규명

**한 내용**: 부록 2에서 견고하다고 확인된 2017 계수가 "사드" 때문이 아니라 **2017년 반도체 슈퍼사이클**(반도체·디스플레이·컴퓨터·통신기기·전지 5개 업종의 ICR 급등) 때문이라는 가설을 검증한다.

**왜 사드가 아닌가**: 사드는 대중국 수출·관광 의존 업종을 때린 사건이라 노출도(단기부채비중)와 연결될 논리적 이유가 없다. 반면 2017년은 실제로 전세계 반도체 슈퍼사이클이 있었던 해이고, 이 5개 업종은 원래 노출도가 낮은(부채가 적은) 업종들이라 우연히 "저노출도-ICR급등"이 겹쳐 노출도와 ICR변화 사이에 가짜 음의 상관을 만들 수 있다.

**과정**: 5개 업종의 2016→2017 실제 ICR을 원자료에서 직접 확인하고, 이 5개를 추가로 뺀 34업종 표본으로 2017 계수를 재추정한다.


In [23]:
TECH_GROUP = ["I3102", "I3103", "I3104", "I3105", "I3108"]  # 반도체·디스플레이·컴퓨터·통신기기·전지

print("1. 전자·반도체 5개 업종 2016->2017 ICR 실제 값")
for code_ in TECH_GROUP:
    name = industry_names_app.get(code_, "?")
    icr16 = profit_data.get((code_, 2016, "이자보상비율"))
    icr17 = profit_data.get((code_, 2017, "이자보상비율"))
    print(f"{code_} {name}: 2016={icr16:.1f}, 2017={icr17:.1f}, 변화={icr17-icr16:+.1f}")


1. 전자·반도체 5개 업종 2016->2017 ICR 실제 값
I3102 반도체: 2016=1161.8, 2017=4759.9, 변화=+3598.1
I3103 디스플레이: 2016=1584.1, 2017=4173.6, 변화=+2589.5
I3104 컴퓨터: 2016=427.3, 2017=860.5, 변화=+433.2
I3105 통신기기: 2016=2015.0, 2017=5077.8, 변화=+3062.8
I3108 전지: 2016=-2655.1, 2017=953.8, 변화=+3608.9


In [24]:
print()
print("2. 표본별 2017 계수 비교")
excl_tobacco_app = pre_app[pre_app["업종코드"] != EXCLUDE_OUTLIER_APP]
b0, p0 = get_2017_coef(excl_tobacco_app)
print(f"담배 제외(39업종): beta={b0:.2f}, p={p0:.3f}")

excl_tobacco_tech_app = pre_app[~pre_app["업종코드"].isin([EXCLUDE_OUTLIER_APP] + TECH_GROUP)]
n_industries_app = excl_tobacco_tech_app["업종코드"].nunique()
b1, p1 = get_2017_coef(excl_tobacco_tech_app)
print(f"담배+전자반도체5개 제외({n_industries_app}업종): beta={b1:.2f}, p={p1:.3f}")



2. 표본별 2017 계수 비교
담배 제외(39업종): beta=-117.64, p=0.014


담배+전자반도체5개 제외(34업종): beta=-8.83, p=0.554


In [25]:
print()
print("3. 노출도 vs 2016->2017 ICR변화 그룹비교")
delta = []
for code_ in industry_codes_app2:
    icr16 = profit_data.get((code_, 2016, "이자보상비율"))
    icr17 = profit_data.get((code_, 2017, "이자보상비율"))
    if icr16 is None or icr17 is None:
        continue
    delta.append({"업종코드": code_, "업종": industry_names_app[code_], "노출도": exposure_map_app[code_], "d_icr": icr17 - icr16})
delta_df = pd.DataFrame(delta)
delta_df = delta_df[delta_df["업종코드"] != EXCLUDE_OUTLIER_APP]

tech = delta_df[delta_df["업종코드"].isin(TECH_GROUP)]
rest = delta_df[~delta_df["업종코드"].isin(TECH_GROUP)]
print(f"전자·반도체(n={len(tech)}): 노출도 평균={tech['노출도'].mean():.2f}, ICR변화 중앙값={tech['d_icr'].median():.1f}")
print(f"그 외(n={len(rest)}): 노출도 평균={rest['노출도'].mean():.2f}, ICR변화 중앙값={rest['d_icr'].median():.1f}")

corr = delta_df["노출도"].corr(delta_df["d_icr"])
corr_no_tech = rest["노출도"].corr(rest["d_icr"])
print(f"Pearson 상관(담배제외 39업종 전체): {corr:.3f}")
print(f"Pearson 상관(전자·반도체도 제외 34업종): {corr_no_tech:.3f}")



3. 노출도 vs 2016->2017 ICR변화 그룹비교
전자·반도체(n=5): 노출도 평균=5.15, ICR변화 중앙값=3062.8
그 외(n=34): 노출도 평균=11.47, ICR변화 중앙값=-76.9
Pearson 상관(담배제외 39업종 전체): -0.519
Pearson 상관(전자·반도체도 제외 34업종): -0.131


**결과**:

| 업종 | 2016 ICR | 2017 ICR | 변화 |
|---|---|---|---|
| 반도체 | 1,161.8 | 4,759.9 | +3,598.1 |
| 디스플레이 | 1,584.1 | 4,173.6 | +2,589.5 |
| 통신기기 | 2,015.0 | 5,077.8 | +3,062.8 |
| 전지 | -2,655.1 | 953.8 | +3,608.9 |

담배 제외(39업종) β=-117.64,p=0.014 → **담배+전자반도체5개 제외(34업종) β=-8.83, p=0.554로 붕괴**. Pearson 상관도 -0.519 → -0.131로 무너짐.

**최종 결론**: 2017 계수는 사드가 아니라 **반도체 슈퍼사이클**이 원인이다 — 저노출도 업종(반도체 계열)이 우연히 그 해 호황과 겹쳐서 노출도-ICR 사이에 가짜 음의 상관을 만든 것. 노출도가 ICR에 영향을 준 게 아니라 반도체 호황이라는 제3의 요인이 두 변수 모두와 상관된 교란(confounding) 사례다.

**논문에 쓸 서술(팀원 제안)**: "2017년 사전추세 이벤트스터디에서 노출도 계수가 유의한 음수(β=-116.7, p=0.028)로 나타났다. 이는 2017년 반도체 호황에 따른 전자·반도체 계열 업종의 이자보상비율 급등에 기인한 것으로, 해당 업종들은 단기부채 비중이 낮아 노출도 계수를 음의 방향으로 끌어당겼다. 실제로 전자·반도체 5개 업종을 제외하면 계수는 -4.5(p=0.804)로 소멸한다."

**더 중요한 함의**: 이건 2017년만의 문제가 아닐 수 있다 — 반도체 사이클 더미를 통제변수로 넣기로 이미 계획돼 있는데, 이 결과가 그 통제가 왜 필수인지에 대한 직접 증거다. 2022~2023년 반도체 다운턴도 같은 방식으로 긴축기 계수를 오염시킬 수 있다. **다음 순서로 자연스러운 것**: 반도체 실물지표(이미 확보된 생산·출하·재고·가동률 지수)를 통제변수로 넣기 전/후로 사전추세 F값이 어떻게 바뀌는지 확인.
